# Exploratory Data Analysis - CICIDS2017
Run `python -m src.data_preprocessing` first so that `data/processed/cicids2017_clean.csv.gz` exists.
Open this notebook from the project root in VS Code (select the project's `.venv` as the kernel).
All outputs are generated when you run the cells; nothing here is pre-filled.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import PROCESSED_FILE, PREPROCESSING_REPORT, load_json
sns.set_theme(style="whitegrid")

df = pd.read_csv(PROCESSED_FILE)
print(df.shape)
df.head()

## 1. Class distribution (class imbalance)

In [ ]:
counts = df["Label"].value_counts()
display(counts.to_frame("records"))
counts.plot.barh(figsize=(7, 3.5), title="Records per class (after per-class capping)")
report = load_json(PREPROCESSING_REPORT) or {}
print("Before capping:", report.get("class_counts_after_dedup_before_cap"))

## 2. Missing and infinite values
Infinite values were converted to NaN in cleaning; the model pipeline imputes them with training medians.

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False).to_frame("missing")

## 3. Feature statistics

In [ ]:
df.describe().T[["mean", "std", "min", "50%", "max"]].head(25)

## 4. Skewness and outliers
Network features are heavily right-skewed. The pipeline clips every feature to its 0.5-99.5 training percentiles.

In [ ]:
feature = "Flow Duration" if "Flow Duration" in df.columns else df.columns[0]
low, high = df[feature].quantile([0.005, 0.995])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(x=df[feature], ax=axes[0], color="#5B8DEF"); axes[0].set_title(f"{feature} - raw")
sns.boxplot(x=df[feature].clip(low, high), ax=axes[1], color="#46C2A5"); axes[1].set_title("after percentile clipping")
plt.tight_layout()

## 5. How do attack types differ?
Median of a few features per class.

In [ ]:
cols = [c for c in ["Flow Duration", "Total Fwd Packets", "Total Backward Packets", "Flow Bytes/s", "Flow Packets/s"] if c in df.columns]
df.groupby("Label")[cols].median().round(2)